#  LOGISCALE BIGDATA — WEEK 1

In [ ]:
"""
================================================================
 LOGISCALE BIGDATA — WEEK 1
 Focus Area : Environment Setup, Data Ingestion & Cleaning
 Dataset    : DataCoSupplyChainDataset.csv (180,519 rows)

 Tasks this week:
   1. Setup local Spark environment
   2. Load dataset using PySpark
   3. Benchmark Pandas vs PySpark
   4. Perform data quality audit
   5. Clean and transform data
   6. Export cleaned dataset

 Outcome:
   - Clean dataset ready for analysis
   - Performance comparison insights
================================================================
"""

## SECTION 1: IMPORTS & LOGGING SETUP

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import time
import os
import logging

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  %(levelname)s  %(message)s")
log = logging.getLogger("week1")

##  SECTION 2: SPARK SESSION & DATA LOADING

In [2]:
import logging
from pyspark.sql import SparkSession

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("LogiScale")

log.info("Initialising Spark session...")

spark = SparkSession.builder \
    .appName("LogiScale-Week1-Ingestion") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
log.info(f"Spark version: {spark.version} | Ready")

CSV_PATH = "C:/Users/Pravith Kumar J/Downloads/Logstic data set/DataCoSupplyChainDataset.csv"
PARQUET_OUT = "dataco_clean.parquet"

df = spark.read.csv(CSV_PATH, header=True, inferSchema=True)

df.show(5)
df.printSchema()

2026-03-26 11:36:29,008  INFO  Initialising Spark session...
2026-03-26 11:36:51,100  INFO  Spark version: 4.1.1 | Ready


+--------+------------------------+-----------------------------+-----------------+------------------+----------------+------------------+-----------+--------------+-------------+----------------+--------------+--------------+-----------+--------------+-----------------+----------------+--------------+--------------------+----------------+-------------+---------------+-----------+------------+------------+----------+-------------+-----------------+-----------------------+--------+----------------------+-------------------+------------------------+-------------+------------------------+-----------------------+-------------------+------+----------------+----------------------+--------------+---------------+---------------+-------------+---------------+-------------------+-------------------+--------------------+------------+-------------+--------------+--------------------------+--------------+
|    Type|Days for shipping (real)|Days for shipment (scheduled)|Benefit per order|Sales per c

## SECTION 3: BENCHMARK — PANDAS vs PYSPARK

In [3]:
import time
import pandas as pd
import pyspark.sql.functions as F

log.info("\n" + "="*55)
log.info("BENCHMARK: Pandas vs PySpark — Data Loading")
log.info("="*55)

t0 = time.time()
df_pd = pd.read_csv(CSV_PATH, encoding="cp1252")
pandas_load = round(time.time() - t0, 3)

t0 = time.time()
_ = df_pd.groupby("Order Region")["Days for shipping (real)"].mean()
pandas_agg = round(time.time() - t0, 3)

t0 = time.time()
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "false") \
    .csv(CSV_PATH)

df_raw.cache()
row_count = df_raw.count()

spark_load = round(time.time() - t0, 3)

t0 = time.time()
_ = df_raw.groupBy("Order Region") \
          .agg(F.avg("Days for shipping (real)")).collect()
spark_agg = round(time.time() - t0, 3)

print("\n" + "─"*55)
print(f"  {'Tool':<15} {'Load Time':>12} {'Agg Time':>12} {'Rows':>10}")
print("─"*55)
print(f"  {'Pandas':<15} {pandas_load:>11}s {pandas_agg:>11}s {len(df_pd):>10,}")
print(f"  {'PySpark':<15} {spark_load:>11}s {spark_agg:>11}s {row_count:>10,}")
print("─"*55)
print("  Verdict: At 10x data scale, PySpark advantage compounds.")
print("  PySpark uses distributed memory — Pandas would crash.")
print("─"*55 + "\n")

2026-03-26 11:38:05,751  INFO  
2026-03-26 11:38:05,754  INFO  BENCHMARK: Pandas vs PySpark — Data Loading
2026-03-26 11:38:05,757  INFO  =======================================================



───────────────────────────────────────────────────────
  Tool               Load Time     Agg Time       Rows
───────────────────────────────────────────────────────
  Pandas                4.526s       0.078s    180,519
  PySpark              14.278s       4.281s    180,519
───────────────────────────────────────────────────────
  Verdict: At 10x data scale, PySpark advantage compounds.
  PySpark uses distributed memory — Pandas would crash.
───────────────────────────────────────────────────────



## SECTION 4: DATA QUALITY AUDIT

In [4]:
from pyspark.sql.types import StringType

log.info("\nRunning data quality audit...")

total_rows = df.count()

null_counts = []

for col_name, dtype in df.dtypes:
    if dtype == 'string':
        n = df.filter(F.col(col_name).isNull() | (F.col(col_name) == "")).count()
    else:
        n = df.filter(F.col(col_name).isNull()).count()

    if n > 0:
        null_counts.append((col_name, n, round(n/total_rows*100,2)))

print("\n  NULL / EMPTY COUNTS:")
print(f"  {'Column':<35} {'Nulls':>8} {'Pct':>8}")
print("  " + "-"*55)

for col_name, n, pct in null_counts:
    print(f"  {col_name:<35} {n:>8,} {pct:>7}%")



2026-03-26 11:39:36,552  INFO  
Running data quality audit...



  NULL / EMPTY COUNTS:
  Column                                 Nulls      Pct
  -------------------------------------------------------
  Customer Lname                             8     0.0%
  Customer Zipcode                           3     0.0%
  Order Zipcode                        155,679   86.24%
  Product Description                  180,519   100.0%


## SECTION 5: DATA CLEANING & TRANSFORMATION

In [6]:


from pyspark.sql.types import DoubleType, IntegerType

log.info("Cleaning data and casting types...")

df_clean = df \
    .filter(F.col("Delivery Status") != "Shipping canceled") \
    .filter(F.col("Customer Id").isNotNull()) \
    .filter(F.col("Sales").isNotNull()) \
    .filter(F.col("Sales").cast(DoubleType()).isNotNull()) \
    .withColumn(
        "order_date",
        F.to_date(
            F.split(F.col("order date (DateOrders)"), " ")[0],
            "M/d/yyyy"
        )
    ) \
    .withColumn(
        "shipping_date",
        F.to_date(
            F.split(F.col("shipping date (DateOrders)"), " ")[0],
            "M/d/yyyy"
        )
    ) \
    .withColumn(
        "delay_days",
        F.col("Days for shipping (real)").cast(IntegerType()) -
        F.col("Days for shipment (scheduled)").cast(IntegerType())
    ) \
    .withColumn("sales",         F.col("Sales").cast(DoubleType())) \
    .withColumn("order_profit",  F.col("Order Profit Per Order").cast(DoubleType())) \
    .withColumn("discount_rate", F.col("Order Item Discount Rate").cast(DoubleType())) \
    .withColumn("quantity",      F.col("Order Item Quantity").cast(IntegerType())) \
    .withColumn("profit_ratio",  F.col("Order Item Profit Ratio").cast(DoubleType())) \
    .withColumn("late_risk",     F.col("Late_delivery_risk").cast(IntegerType())) \
    .withColumn("year",          F.year("order_date")) \
    .withColumn("month",         F.month("order_date")) \
    .withColumn("quarter",       F.quarter("order_date")) \
    .withColumn("week_of_year",  F.weekofyear("order_date"))

# Drop PII columns (use original names)
pii_cols = [
    "Customer Email", "Customer Password",
    "Customer Street", "Customer Lname", "Customer Fname"
]

existing_pii = [c for c in pii_cols if c in df_clean.columns]
df_clean = df_clean.drop(*existing_pii)

# Final count
clean_count = df_clean.count()

log.info(f"Clean rows: {clean_count:,} (removed {total_rows - clean_count:,} rows)")

2026-03-26 11:46:37,944  INFO  Cleaning data and casting types...
2026-03-26 11:46:40,573  INFO  Clean rows: 172,765 (removed 7,754 rows)


## SECTION 6: DATA EXPORT (CSV VIA PANDAS)

In [7]:
import os

os.makedirs("C:/temp", exist_ok=True)

spark.conf.set("spark.local.dir", "C:/temp")
spark.conf.set("java.io.tmpdir", "C:/temp")

pdf = df_clean.toPandas()

pdf.to_csv(
    r"C:\Users\Pravith Kumar J\Downloads\final_output.csv",
    index=False
)

print("CSV saved successfully")

CSV saved successfully


## SECTION 7: WEEK 1 SUMMARY

In [8]:
print("\n" + "="*55)
print("  WEEK 1 COMPLETE — SUMMARY")
print("="*55)

print(f"  Raw rows loaded        : {df_raw.count():,}")
print(f"  Clean rows             : {df_clean.count():,}")
print(f"  Rows removed           : {df_raw.count() - df_clean.count():,} (duplicates/nulls)")

try:
    print(f"  Pandas load time       : {pandas_load}s")
except:
    pass

try:
    print(f"  PySpark load time      : {spark_load}s")
except:
    pass

try:
    print(f"  Late delivery risk     : {late_risk/df_raw.count()*100:.1f}% of orders")
except:
    print("  Late delivery risk     : Not calculated")

print(f"  Output file            : C:\\Users\\Pravith Kumar J\\Downloads\\final_output.csv")

print("="*55)
print("\n  Next step: EDA & Insights")


  WEEK 1 COMPLETE — SUMMARY
  Raw rows loaded        : 180,519
  Clean rows             : 172,765
  Rows removed           : 7,754 (duplicates/nulls)
  Pandas load time       : 4.526s
  PySpark load time      : 14.278s
  Late delivery risk     : Not calculated
  Output file            : C:\Users\Pravith Kumar J\Downloads\final_output.csv

  Next step: EDA & Insights
